# 04 — Model Training and Evaluation

**Project**: LLM-Based Startup Risk Profiling and Funding Outcome Prediction

This notebook covers:
1. Train/val/test split
2. Baseline model (Logistic Regression)
3. Decision Tree, Random Forest
4. XGBoost, LightGBM
5. Optuna hyperparameter optimization
6. Cross-validation
7. Model comparison and selection

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_loader import load_and_preprocess, load_config
from src.train import split_data, train_all_models, optimize_xgboost_with_optuna, cross_validate_models, save_model
from src.evaluate import compare_models, evaluate_model

In [ ]:
config = load_config()
df = load_and_preprocess(config['dataset']['file'])
print(f'Shape: {df.shape}')

## 1. Split Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

## 2. Train All Models

In [ ]:
trained = train_all_models(X_train, y_train, X_val, y_val, use_llm=False)

## 3. Optuna Optimization (XGBoost)

In [ ]:
best_xgb, best_params = optimize_xgboost_with_optuna(
    X_train, y_train, X_val, y_val, n_trials=50, use_llm=False
)
if best_xgb:
    trained['XGBoost (Optuna)'] = best_xgb
    print(f'Best params: {best_params}')

## 4. Cross-Validation

In [ ]:
X = df.drop(columns=['funding_success'])
y = df['funding_success']
cv_results = cross_validate_models(X, y, cv_folds=5, use_llm=False)

## 5. Compare on Test Set

In [ ]:
comparison = compare_models(trained, X_test, y_test)
comparison

## 6. Save Best Model

In [ ]:
# Select best model by ROC-AUC on test
best_name = comparison.loc[comparison['roc_auc'].idxmax(), 'Model']
print(f'Best model: {best_name}')
save_model(trained[best_name])